In [2]:
import os
from pathlib import Path
from PIL import Image
import numpy as np
from tqdm import tqdm
import replicate
import requests
from io import BytesIO
import json

class BBoxPreservingBackgroundReplacer:
    """
    Класс для замены фона с сохранением позиций объектов и разметки YOLO
    """
    
    def __init__(self, api_token):
        """Инициализация с токеном Replicate API"""
        os.environ["REPLICATE_API_TOKEN"] = api_token
        self.backgrounds = []
        self.removed_bg_cache = {}
    
    def load_existing_backgrounds(self, backgrounds_folder):
        """
        Загрузка готовых фонов из папки
        """
        bg_folder = Path(backgrounds_folder)
        if not bg_folder.exists():
            print(f"❌ Папка {backgrounds_folder} не найдена")
            return []
        
        bg_files = sorted(list(bg_folder.glob("*.jpg")) + list(bg_folder.glob("*.png")))
        
        print(f"📁 Загрузка фонов из {backgrounds_folder}")
        for bg_path in bg_files:
            bg_image = Image.open(bg_path)
            self.backgrounds.append(bg_image)
            print(f"  ✅ Загружен: {bg_path.name} (размер: {bg_image.size})")
        
        print(f"📊 Загружено фонов: {len(self.backgrounds)}")
        return self.backgrounds
    
    def load_yolo_annotations(self, annotation_path):
        """
        Загрузка YOLO аннотаций
        """
        annotations = []
        if Path(annotation_path).exists():
            with open(annotation_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        annotations.append({
                            'class_id': int(parts[0]),
                            'x_center': float(parts[1]),
                            'y_center': float(parts[2]),
                            'width': float(parts[3]),
                            'height': float(parts[4])
                        })
        return annotations
    
    def save_yolo_annotations(self, annotations, output_path):
        """Сохранение YOLO аннотаций"""
        with open(output_path, 'w') as f:
            for ann in annotations:
                f.write(f"{ann['class_id']} {ann['x_center']} {ann['y_center']} {ann['width']} {ann['height']}\n")
    
    def remove_background_replicate(self, image_path):
        """Удаление фона через Replicate с кэшированием"""
        img_str = str(image_path)
        
        # Проверяем кэш
        if img_str in self.removed_bg_cache:
            return self.removed_bg_cache[img_str]
        
        # Удаляем фон
        output = replicate.run(
            "cjwbw/rembg:fb8af171cfa1616ddcf1242c093f9c46bcada5ad4cf6f2fbe8b81b330ec5c003",
            input={
                "image": open(image_path, "rb"),
                "alpha_matting": True,
                "alpha_matting_foreground_threshold": 240,
                "alpha_matting_background_threshold": 50,
                "alpha_matting_erode_size": 10
            }
        )
        
        response = requests.get(output)
        tool_img = Image.open(BytesIO(response.content)).convert('RGBA')
        
        # Сохраняем в кэш
        self.removed_bg_cache[img_str] = tool_img
        
        return tool_img
    
    def adapt_background_to_image_size(self, background_img, target_size, method='smart_crop'):
        """
        Адаптация фона под размер целевого изображения
        """
        bg_width, bg_height = background_img.size
        target_width, target_height = target_size
        
        if method == 'stretch':
            # Простое растягивание
            return background_img.resize(target_size, Image.Resampling.LANCZOS)
        
        elif method == 'smart_crop':
            # Умная адаптация
            bg_ratio = bg_width / bg_height
            target_ratio = target_width / target_height
            
            if abs(bg_ratio - target_ratio) < 0.1:
                return background_img.resize(target_size, Image.Resampling.LANCZOS)
            
            scale = max(target_width / bg_width, target_height / bg_height)
            
            new_width = int(bg_width * scale)
            new_height = int(bg_height * scale)
            resized_bg = background_img.resize((new_width, new_height), Image.Resampling.LANCZOS)
            
            left = (new_width - target_width) // 2
            top = (new_height - target_height) // 2
            right = left + target_width
            bottom = top + target_height
            
            return resized_bg.crop((left, top, right, bottom))
        
        elif method == 'tile':
            # Замощение
            result = Image.new('RGB', target_size)
            
            tiles_x = (target_width // bg_width) + 2
            tiles_y = (target_height // bg_height) + 2
            
            for i in range(tiles_x):
                for j in range(tiles_y):
                    x = i * bg_width
                    y = j * bg_height
                    result.paste(background_img, (x, y))
            
            return result.crop((0, 0, target_width, target_height))
    
    def replace_background_preserve_position(self, 
                                            original_image_path, 
                                            background_img,
                                            output_path,
                                            adaptation_method='smart_crop'):
        """
        Замена фона с сохранением позиций объектов
        """
        original = Image.open(original_image_path)
        original_size = original.size
        
        tools_transparent = self.remove_background_replicate(original_image_path)
        
        if tools_transparent.size != original_size:
            tools_transparent = tools_transparent.resize(original_size, Image.Resampling.LANCZOS)
        
        background_adapted = self.adapt_background_to_image_size(
            background_img, 
            original_size,
            method=adaptation_method
        )
        background_adapted = background_adapted.convert('RGBA')
        
        final_image = background_adapted.copy()
        final_image.paste(tools_transparent, (0, 0), tools_transparent)
        
        final_image = final_image.convert('RGB')
        final_image.save(output_path, quality=95)
        
        return final_image
    
    def process_dataset_with_annotations(self, 
                                        images_folder,
                                        annotations_folder,
                                        output_images_folder,
                                        output_annotations_folder,
                                        backgrounds_folder="backgrounds",
                                        adaptation_method='smart_crop'):
        """
        Обработка всего датасета с сохранением YOLO разметки
        """
        
        images_path = Path(images_folder)
        annotations_path = Path(annotations_folder)
        output_imgs_path = Path(output_images_folder)
        output_anns_path = Path(output_annotations_folder)
        
        output_imgs_path.mkdir(parents=True, exist_ok=True)
        output_anns_path.mkdir(parents=True, exist_ok=True)
        
        image_files = list(images_path.glob("*.jpg")) + list(images_path.glob("*.png"))
        
        if not image_files:
            print(f"❌ Изображения не найдены в {images_folder}")
            return
        
        print(f"📊 Найдено изображений: {len(image_files)}")
        
        # Загружаем фоны если не загружены
        if not self.backgrounds:
            self.load_existing_backgrounds(backgrounds_folder)
        
        if not self.backgrounds:
            print("❌ Фоны не найдены! Положите фоны в папку", backgrounds_folder)
            return
        
        print(f"\n🔧 Обработка с {len(self.backgrounds)} фонами...")
        print(f"📐 Метод адаптации размера: {adaptation_method}")
        print("=" * 50)
        
        total_combinations = len(image_files) * len(self.backgrounds)
        pbar = tqdm(total=total_combinations, desc="Создание комбинаций")
        
        for img_file in image_files:
            img_name = img_file.stem
            
            ann_file = annotations_path / f"{img_name}.txt"
            
            annotations = []
            if ann_file.exists():
                annotations = self.load_yolo_annotations(ann_file)
            
            for bg_idx, background in enumerate(self.backgrounds, 1):
                output_img_name = f"{img_name}_bg{bg_idx:02d}.jpg"
                output_ann_name = f"{img_name}_bg{bg_idx:02d}.txt"
                
                output_img_path = output_imgs_path / output_img_name
                output_ann_path = output_anns_path / output_ann_name
                
                try:
                    self.replace_background_preserve_position(
                        img_file,
                        background,
                        output_img_path,
                        adaptation_method=adaptation_method
                    )
                    
                    if annotations:
                        self.save_yolo_annotations(annotations, output_ann_path)
                    
                    pbar.update(1)
                    
                except Exception as e:
                    print(f"\n❌ Ошибка обработки {img_name}: {str(e)}")
                    pbar.update(1)
                    continue
        
        pbar.close()
        print(f"\n✅ Готово!")
        print(f"📁 Изображения: {output_images_folder}")
        print(f"📁 Аннотации: {output_annotations_folder}")

print("✅ Класс BBoxPreservingBackgroundReplacer загружен!")

✅ Класс BBoxPreservingBackgroundReplacer загружен!


In [12]:
# НАСТРОЙКИ
REPLICATE_TOKEN = "YOUR_REPLICATE_API_TOKEN_HERE"

# Папки
IMAGES_FOLDER = "dataset/selected_dataset/images"              # Исходные изображения
ANNOTATIONS_FOLDER = "dataset/selected_dataset/labels"         # YOLO разметка
BACKGROUNDS_FOLDER = "backgrounds"            # Ваши готовые фоны 1024x1024
OUTPUT_IMAGES = "dataset/images_new_bg"       
OUTPUT_ANNOTATIONS = "dataset/labels_new_bg"  

# Создаем обработчик
processor = BBoxPreservingBackgroundReplacer(api_token=REPLICATE_TOKEN)

# Загружаем ваши готовые фоны
processor.load_existing_backgrounds(BACKGROUNDS_FOLDER)

# Обрабатываем датасет
processor.process_dataset_with_annotations(
    images_folder=IMAGES_FOLDER,
    annotations_folder=ANNOTATIONS_FOLDER,
    output_images_folder=OUTPUT_IMAGES,
    output_annotations_folder=OUTPUT_ANNOTATIONS,
    backgrounds_folder=BACKGROUNDS_FOLDER,
    adaptation_method='smart_crop'  # Метод адаптации размера
)

📁 Загрузка фонов из backgrounds
  ✅ Загружен: background_01.png (размер: (1024, 1024))
  ✅ Загружен: background_02.jpg (размер: (1024, 1024))
  ✅ Загружен: background_03.jpg (размер: (1024, 1024))
  ✅ Загружен: background_04.jpg (размер: (1024, 1024))
  ✅ Загружен: background_05.jpg (размер: (1024, 1024))
📊 Загружено фонов: 5
📊 Найдено изображений: 166

🔧 Обработка с 5 фонами...
📐 Метод адаптации размера: smart_crop



Создание комбинаций: 100%|████████████████████| 830/830 [46:40<00:00,  3.37s/it]


✅ Готово!
📁 Изображения: dataset/images_new_bg
📁 Аннотации: dataset/labels_new_bg


In [29]:
# ========== ЯЧЕЙКА 1: ГЕНЕРАЦИЯ ФОНОВ ==========
# Запустите эту ячейку первой, чтобы сгенерировать и посмотреть фоны

REPLICATE_TOKEN = "YOUR_REPLICATE_API_TOKEN_HERE"  # ← Вставьте ваш токен!

# НАСТРОЙТЕ СВОИ ПРОМПТЫ (5 штук):
BACKGROUND_PROMPTS = [
 # Фон 1
"top-down overhead photo of an aviation maintenance workbench surface, red tool cart top with black foam organizer impressions, completely empty, no tools, no instruments, no objects, flat clean surface only, realistic photography, high detail"]

# ГЕНЕРАЦИЯ
processor = MultiBackgroundReplacer(api_token=REPLICATE_TOKEN)

print("🎨 Генерирую фоны...")
backgrounds = processor.generate_backgrounds(
    prompts=BACKGROUND_PROMPTS,
    size=(1024, 1024),
    save_folder="backgrounds"  # Сохранятся сюда
)

print(f"\n✅ Готово! Сгенерировано {len(backgrounds)} фонов")
print("📁 Посмотрите фоны в папке 'backgrounds/'")
print("👍 Если фоны устраивают - запустите ячейку 2")

🎨 Генерирую фоны...
🎨 Генерация 1 фонов...

[Фон 1/1]
Промпт: top-down overhead photo of an aviation maintenance workbench surface, red tool cart top with black f...
✅ Фон 1 сгенерирован и сохранен: backgrounds/background_01.jpg

✨ Сгенерировано фонов: 1/1

✅ Готово! Сгенерировано 1 фонов
📁 Посмотрите фоны в папке 'backgrounds/'
👍 Если фоны устраивают - запустите ячейку 2


In [32]:
# ========== ЯЧЕЙКА 3: ПРИМЕНЕНИЕ К ИНСТРУМЕНТАМ ==========

INPUT_FOLDER = "dataset/tools"  # Ваша папка с инструментами
OUTPUT_FOLDER = "dataset/tools_new_bg"  # Куда сохранять результаты
REPLICATE_TOKEN = "YOUR_REPLICATE_API_TOKEN_HERE"  # Получить на https://replicate.com/account/api-tokens
# Создаем новый процессор и загружаем фоны
processor = MultiBackgroundReplacer(api_token=REPLICATE_TOKEN)

print("📁 Загружаю фоны Аэрофлота...")
processor.load_existing_backgrounds("backgrounds_aeroflot")

if processor.backgrounds:
    print(f"✅ Загружено {len(processor.backgrounds)} фонов")
    processor.process_all_combinations(INPUT_FOLDER, OUTPUT_FOLDER)
else:
    print("❌ Фоны не найдены!")

📁 Загружаю фоны Аэрофлота...
📁 Загрузка фонов из backgrounds_aeroflot
  ✅ Загружен: background_01.jpg
  ✅ Загружен: background_02.jpg
  ✅ Загружен: background_03.jpg
  ✅ Загружен: background_04.jpg
  ✅ Загружен: background_05.jpg
📊 Загружено фонов: 5
✅ Загружено 5 фонов

🔧 Обработка инструментов...
Инструментов: 1
Фонов: 5
Всего комбинаций: 5


Создание комбинаций:   0%|                                | 0/5 [00:00<?, ?it/s]


[1/1] Инструмент: DSCN2536.JPG
  → Удаление фона...


Создание комбинаций: 100%|█| 5/5 [00:21<00:00,  4.32s/it, текущий=DSCN2536 + фон

  ✅ Создано 5 вариантов

✨ Готово! Создано комбинаций: 5
📁 Результаты в: dataset/tools_new_bg


In [10]:
from pathlib import Path
import shutil
import random
from collections import defaultdict

def select_dataset_with_classes(images_folder="dataset/images_new_bg",
                                labels_folder="dataset/labels_new_bg",
                                output_folder="selected_dataset",
                                num_per_class=6,  # Теперь 6 вместо 5
                                num_gruppovye=100):
    """
    Отбирает:
    - По 6 картинок каждого из 11 классов инструментов
    - 100 картинок gruppovye
    """
    
    # Ваши классы инструментов
    TOOL_CLASSES = [
        'bokorezy',
        'klyuch_rozhkovyi_nakidnoj',
        'kolovorot',
        'otkryvashka',
        'otvertka_krest',
        'otvertka_minus',
        'otvertka_plus',
        'passatizhi',
        'passatizhi_kontrov',
        'razvodnoj_klyuch',
        'shernitsa'
    ]
    
    images_path = Path(images_folder)
    labels_path = Path(labels_folder)
    output_path = Path(output_folder)
    
    # Создаем выходные папки
    (output_path / "images").mkdir(parents=True, exist_ok=True)
    (output_path / "labels").mkdir(parents=True, exist_ok=True)
    
    print("🔧 ОТБОР ДАТАСЕТА ИНСТРУМЕНТОВ")
    print("=" * 60)
    print(f"📊 Классы инструментов: {len(TOOL_CLASSES)}")
    print(f"📌 Изображений на класс: {num_per_class}")
    print(f"📌 Групповых изображений: {num_gruppovye}")
    print(f"📁 Источник: {images_folder}")
    print(f"📁 Результат: {output_folder}")
    print("=" * 60)
    
    total_copied = 0
    stats = {}
    
    # 1. ОТБОР ОДИНОЧНЫХ ИНСТРУМЕНТОВ (по 6 каждого класса)
    print(f"\n📌 ОТБОР ОДИНОЧНЫХ ИНСТРУМЕНТОВ (по {num_per_class}):")
    print("-" * 40)
    
    for class_name in TOOL_CLASSES:
        # Находим все изображения этого класса
        class_images = []
        
        # Ищем с разными паттернами
        for pattern in [f"{class_name}_*.jpg", f"{class_name}*.jpg", 
                       f"{class_name}_*.JPG", f"{class_name}*.JPG"]:
            class_images.extend(list(images_path.glob(pattern)))
        
        # Убираем дубликаты и gruppovye
        class_images = list(set(class_images))
        class_images = [img for img in class_images if 'gruppovye' not in img.stem.lower()]
        
        # Фильтруем только те, у которых есть разметка
        valid_images = []
        for img in class_images:
            label_file = labels_path / f"{img.stem}.txt"
            if label_file.exists():
                valid_images.append(img)
        
        # Отбираем случайные (до num_per_class)
        if valid_images:
            num_to_select = min(num_per_class, len(valid_images))
            selected = random.sample(valid_images, num_to_select)
            
            # Копируем файлы
            for img in selected:
                shutil.copy2(img, output_path / "images" / img.name)
                shutil.copy2(labels_path / f"{img.stem}.txt", 
                           output_path / "labels" / f"{img.stem}.txt")
                total_copied += 1
            
            stats[class_name] = len(selected)
            
            # Статус индикатор
            if len(selected) == num_per_class:
                status = "✅"
            elif len(selected) > 0:
                status = "⚠️"
            else:
                status = "❌"
                
            print(f"  {status} {class_name:30} отобрано: {len(selected)}/{num_per_class} (доступно: {len(valid_images)})")
        else:
            stats[class_name] = 0
            print(f"  ❌ {class_name:30} НЕ НАЙДЕНО!")
    
    # 2. ОТБОР ГРУППОВЫХ ИЗОБРАЖЕНИЙ
    print(f"\n📌 ОТБОР ГРУППОВЫХ ИЗОБРАЖЕНИЙ ({num_gruppovye}):")
    print("-" * 40)
    
    # Находим все gruppovye
    gruppovye_images = []
    for pattern in ["*gruppovye*.jpg", "*gruppovye*.JPG", "*group*.jpg", "*group*.JPG"]:
        gruppovye_images.extend(list(images_path.glob(pattern)))
    
    # Убираем дубликаты
    gruppovye_images = list(set(gruppovye_images))
    
    # Фильтруем с разметкой
    valid_gruppovye = []
    for img in gruppovye_images:
        label_file = labels_path / f"{img.stem}.txt"
        if label_file.exists():
            valid_gruppovye.append(img)
    
    # Отбираем случайных
    if valid_gruppovye:
        num_to_select = min(num_gruppovye, len(valid_gruppovye))
        selected_gruppovye = random.sample(valid_gruppovye, num_to_select)
        
        # Копируем
        for img in selected_gruppovye:
            shutil.copy2(img, output_path / "images" / img.name)
            shutil.copy2(labels_path / f"{img.stem}.txt", 
                       output_path / "labels" / f"{img.stem}.txt")
            total_copied += 1
        
        status = "✅" if len(selected_gruppovye) == num_gruppovye else "⚠️"
        print(f"  {status} Групповые (gruppovye): отобрано {len(selected_gruppovye)}/{num_gruppovye} (доступно: {len(valid_gruppovye)})")
        stats['gruppovye'] = len(selected_gruppovye)
    else:
        print(f"  ❌ Групповые изображения НЕ НАЙДЕНЫ!")
        stats['gruppovye'] = 0
    
    # 3. ИТОГОВАЯ СТАТИСТИКА
    print("\n" + "=" * 60)
    print("📊 ИТОГОВАЯ СТАТИСТИКА:")
    print("-" * 40)
    
    total_single = sum(stats.get(c, 0) for c in TOOL_CLASSES)
    total_group = stats.get('gruppovye', 0)
    
    print(f"  Одиночных инструментов: {total_single}")
    print(f"    • Ожидалось: {len(TOOL_CLASSES) * num_per_class} (11 классов × {num_per_class})")
    print(f"    • Получено: {total_single}")
    
    # Детализация по классам если не все набрали
    if total_single < len(TOOL_CLASSES) * num_per_class:
        print("\n  Детализация недостающих:")
        for class_name in TOOL_CLASSES:
            count = stats.get(class_name, 0)
            if count < num_per_class:
                print(f"    • {class_name}: не хватает {num_per_class - count}")
    
    print(f"\n  Групповых изображений: {total_group}")
    print(f"    • Ожидалось: {num_gruppovye}")
    print(f"    • Получено: {total_group}")
    
    print(f"\n  📁 ВСЕГО СКОПИРОВАНО: {total_copied} пар файлов")
    print(f"  📂 Сохранено в: {output_path}/")
    
    # Сохраняем статистику в файл
    stats_file = output_path / "selection_stats.txt"
    with open(stats_file, 'w', encoding='utf-8') as f:
        f.write("СТАТИСТИКА ОТБОРА ДАТАСЕТА\n")
        f.write("=" * 40 + "\n\n")
        f.write("ОДИНОЧНЫЕ ИНСТРУМЕНТЫ:\n")
        for class_name in TOOL_CLASSES:
            count = stats.get(class_name, 0)
            f.write(f"  {class_name}: {count}/{num_per_class}\n")
        f.write(f"\nИтого одиночных: {total_single}/{len(TOOL_CLASSES) * num_per_class}\n")
        f.write(f"\nГРУППОВЫЕ: {stats.get('gruppovye', 0)}/{num_gruppovye}\n")
        f.write(f"\nВСЕГО ФАЙЛОВ: {total_copied}\n")
    
    print(f"  📝 Статистика сохранена: {stats_file}")
    
    # Создаем dataset.yaml для YOLO
    yaml_content = f"""# Dataset configuration for YOLO
path: {output_path.absolute()}
train: images
val: images

# Number of classes  
nc: 11

# Class names
names: {TOOL_CLASSES}
"""
    
    yaml_file = output_path / "dataset.yaml"
    with open(yaml_file, 'w') as f:
        f.write(yaml_content)
    
    print(f"  📝 Конфигурация YOLO: {yaml_file}")
    
    return stats

# =====================================================
# ЗАПУСК
# =====================================================

# Запуск с 6 изображениями на класс
stats = select_dataset_with_classes(
    images_folder="dataset/images",
    labels_folder="dataset/labels",
    output_folder="selected_dataset",
    num_per_class=6,      # 6 картинок каждого класса
    num_gruppovye=100     # 100 групповых
)

🔧 ОТБОР ДАТАСЕТА ИНСТРУМЕНТОВ
📊 Классы инструментов: 11
📌 Изображений на класс: 6
📌 Групповых изображений: 100
📁 Источник: dataset/images
📁 Результат: selected_dataset

📌 ОТБОР ОДИНОЧНЫХ ИНСТРУМЕНТОВ (по 6):
----------------------------------------
  ✅ bokorezy                       отобрано: 6/6 (доступно: 37)
  ✅ klyuch_rozhkovyi_nakidnoj      отобрано: 6/6 (доступно: 38)
  ✅ kolovorot                      отобрано: 6/6 (доступно: 23)
  ✅ otkryvashka                    отобрано: 6/6 (доступно: 30)
  ✅ otvertka_krest                 отобрано: 6/6 (доступно: 39)
  ✅ otvertka_minus                 отобрано: 6/6 (доступно: 24)
  ✅ otvertka_plus                  отобрано: 6/6 (доступно: 23)
  ✅ passatizhi                     отобрано: 6/6 (доступно: 68)
  ✅ passatizhi_kontrov             отобрано: 6/6 (доступно: 31)
  ✅ razvodnoj_klyuch               отобрано: 6/6 (доступно: 31)
  ✅ shernitsa                      отобрано: 6/6 (доступно: 28)

📌 ОТБОР ГРУППОВЫХ ИЗОБРАЖЕНИЙ (100):
---------